import re
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import matthews_corrcoef, confusion_matrix

In [6]:
path = ("/project/PlatigLab/users/reece/ENCODE-RNA-Binding-Protein-Network-Modeling/analysis/07_reece_biological_validations/1_for_figures/3_Crispr_KD/FINAL_results/final_combined_results.csv") # final results for 10 reads concatinated

In [7]:
table = pd.read_csv(path)

In [8]:
table

,Feature,RBP-IS-KD-Target,Position,cell_line,index,dPSI,CTRL - KD Local SHAP,FDR
0,APEX1_1_binding,APEX1,1,K562,chr12_-_56645928_56645836_56645353_56645170_56...,0.000,0.000000,1.000000
1,APEX1_1_binding,APEX1,1,K562,chr9_+_136665791_136665817_136668240_136668362...,-0.015,0.000000,1.000000
2,APEX1_1_binding,APEX1,1,K562,chr4_-_82373777_82373445_82359639_82359470_823...,-0.001,0.000000,1.000000
3,APEX1_1_binding,APEX1,1,K562,chr19_+_58544472_58545097_58547375_58547511_58...,0.000,0.000000,1.000000
4,APEX1_1_binding,APEX1,1,K562,chr12_-_56645928_56645836_56645353_56645170_56...,0.012,0.000000,0.027580
...,...,...,...,...,...,...,...,...
43325,SRSF7_6_binding,SRSF7,6,HepG2,chr22_+_46352263_46352330_46353766_46353867_46...,-0.028,0.000064,0.821646
43326,SRSF7_6_binding,SRSF7,6,HepG2,chr20_+_33019061_33019115_33019579_33019750_33...,-0.001,0.000070,1.000000
43327,SRSF7_6_binding,SRSF7,6,HepG2,chr21_-_46150250_46150118_46146327_46146265_46...,-0.017,0.000067,1.000000
43328,SRSF7_6_binding,SRSF7,6,HepG2,chr6_-_31541986_31541949_31541327_31541152_315...,-0.003,0.000064,1.000000


## Summarize data (ALL data, NOT per feature)

In [18]:
from scipy.stats import fisher_exact

def summarize_all_data(data):
    cell_lines = ['HepG2', 'K562']
    rows = []

    for cell_line in cell_lines:

        # ------------------------------------------------------------------
        # 1. Subset & significance filter
        # ------------------------------------------------------------------
        subset = data[data['cell_line'] == cell_line].copy()

        subset['Significance'] = (
            (subset['dPSI'].abs() > 0) &
            (subset['FDR'] <= 0.1)
        ).astype(int)

        significant_subset = subset[subset['Significance'] == 1].copy()
        num_significant = len(significant_subset)

        # ------------------------------------------------------------------
        # 2. MCC + Fisher's Exact Test (requires ≥ 10 significant events)
        # ------------------------------------------------------------------

        if num_significant >= 10:
            y_true = np.sign(significant_subset['dPSI'].values).astype(int)
            y_pred = np.sign(significant_subset['CTRL - KD Local SHAP'].values).astype(int)

            nonzero_mask = (y_true != 0) & (y_pred != 0)
            y_true_nz = y_true[nonzero_mask]
            y_pred_nz = y_pred[nonzero_mask]

            mcc = matthews_corrcoef(y_true_nz, y_pred_nz)

            tp = ((y_pred_nz ==  1) & (y_true_nz ==  1)).sum()
            fp = ((y_pred_nz ==  1) & (y_true_nz == -1)).sum()
            fn = ((y_pred_nz == -1) & (y_true_nz ==  1)).sum()
            tn = ((y_pred_nz == -1) & (y_true_nz == -1)).sum()
            odds_ratio, fisher_p = fisher_exact([[tp, fp], [fn, tn]], alternative='two-sided')

            # ------------------------------------------------------------------
            # 3. PSI–SHAP concordance (% of events where signs match)
            # ------------------------------------------------------------------
            concordance = np.mean(y_true_nz == y_pred_nz) * 100

            rows.append({
                'Cell Line': cell_line,
                'Significant Events': num_significant,
                'MCC': mcc,
                'dpsi_dshap_concordance': concordance,
                'FET OR': odds_ratio,
                'FET P-Value': fisher_p,
            })

    return pd.DataFrame(rows).sort_values('MCC', ascending=False).reset_index(drop=True)

In [19]:
all_data = summarize_all_data(table)

In [20]:
all_data.to_csv("all_data.csv")

## Summarize the data per feature

In [21]:
from scipy.stats import fisher_exact

def summarize_mcc(data):
    cell_lines = ['HepG2', 'K562']
    features = data['Feature'].unique()
    rows = []
    
    for feature in features:
        parts = feature.split('_')
        rbp_name = parts[0]
        position = parts[1]
        rbp_label = f"{rbp_name} Position {position}"
        for cell_line in cell_lines:
    
            # ------------------------------------------------------------------
            # 1. Subset & significance filter
            # ------------------------------------------------------------------
            feature_subset = data[
                (data['cell_line'] == cell_line) &
                (data['Feature'] == feature)
            ].copy()
    
            feature_subset['Significance'] = (
                (feature_subset['dPSI'].abs() > 0) &
                (feature_subset['FDR'] <= 0.1)
            ).astype(int)
    
            significant_subset = feature_subset[feature_subset['Significance'] == 1].copy()
            num_significant = len(significant_subset)
    
            # ------------------------------------------------------------------
            # 2. MCC + Fisher's Exact Test (requires ≥ 10 significant events)
            # ------------------------------------------------------------------
    
            if num_significant >= 10:
                # Ground truth  → sign of dPSI   (+1 / -1)
                # Predicted     → sign of SHAP   (+1 / -1)
                y_true = np.sign(significant_subset['dPSI'].values).astype(int)
                y_pred = np.sign(significant_subset['CTRL - KD Local SHAP'].values).astype(int)

                # Drop zeros from both vectors consistently
                nonzero_mask = (y_true != 0) & (y_pred != 0)
                y_true_nz = y_true[nonzero_mask]
                y_pred_nz = y_pred[nonzero_mask]
    
                mcc = matthews_corrcoef(y_true_nz, y_pred_nz)

                # 2x2 contingency table
                # Rows: predicted sign | Cols: true sign
                tp = ((y_pred_nz ==  1) & (y_true_nz ==  1)).sum()
                fp = ((y_pred_nz ==  1) & (y_true_nz == -1)).sum()
                fn = ((y_pred_nz == -1) & (y_true_nz ==  1)).sum()
                tn = ((y_pred_nz == -1) & (y_true_nz == -1)).sum()

                odds_ratio, fisher_p = fisher_exact([[tp, fp], [fn, tn]], alternative='two-sided')

                # ------------------------------------------------------------------
                # 3. PSI–SHAP concordance (% of events where signs match)
                # ------------------------------------------------------------------
                concordance = np.mean(y_true_nz == y_pred_nz) * 100
    
                rows.append({
                    'Feature': rbp_label,
                    'Cell Line': cell_line,
                    'Significant Events': num_significant,
                    'MCC': mcc,
                    'dpsi_dshap_concordance': concordance,
                    'FET OR': odds_ratio,
                    'FET P-Value': fisher_p,
                })
    
    return pd.DataFrame(rows).sort_values('MCC', ascending=False).reset_index(drop=True)

In [22]:
crispr_data = summarize_mcc(table)

In [23]:
crispr_data.to_csv("final_mcc_crispr.csv")

## 2x3 plot of dPSI vs CTRL - KD Local SHAP

In [5]:
# Given a dataframe and RBP -> make a 2x3 plot of dPSI vs CTRL - KD Local SHAP

def make_scatterplot_grid(data, RBP):
    
    # Filter the dataframe for the RBP
    plot_df = data[data["Feature"].str.contains(RBP, na=False)]
    
    # Get unique features and cell lines
    features = sorted(plot_df["Feature"].unique())
    cell_lines = ["HepG2", "K562"]
    
    # Create figure with 2 rows and 6 columns
    fig, axes = plt.subplots(2, 6, figsize=(30, 10), sharex=True, sharey=True)
    
    # Plot for each cell line (row) and feature (column)
    for row_idx, cell_line in enumerate(cell_lines):
        for col_idx, feature in enumerate(features):
            ax = axes[row_idx, col_idx]
            
            # Filter data for this specific feature and cell line
            subset = plot_df[(plot_df["Feature"] == feature) & 
                            (plot_df["cell_line"] == cell_line)]
            
            if len(subset) > 0:
                # Create the scatter plot
                ax.scatter(subset["CTRL - KD Local SHAP"], subset["dPSI"], 
                          alpha=0.5, s=30)
            
            # Add quadrant lines at x=0 and y=0
            ax.axhline(y=0, color='black', linestyle='-', linewidth=1.5)
            ax.axvline(x=0, color='black', linestyle='-', linewidth=1.5)
            
            # Add grid
            ax.grid(True, alpha=0.3)
            
            # Parse feature name (format: RBP_position_binding)
            parts = feature.split('_')
            rbp_name = parts[0]
            position = parts[1]
            title = f"{rbp_name} Pos. {position}"
            
            # Add title with parsed feature name
            ax.set_title(title, fontsize=14, pad=10)
            
            # Increase tick label font size
            ax.tick_params(axis='both', which='major', labelsize=12)
        
        # Add cell line label to the left of the first plot in each row
        axes[row_idx, 0].text(-0.3, 0.5, cell_line, 
                              transform=axes[row_idx, 0].transAxes,
                              fontsize=20, fontweight='bold',
                              rotation=90, va='center', ha='center')
    
    # Add overall title
    fig.suptitle(f"{RBP} - Per Feature Analysis", fontsize=24, y=0.98)
    fig.supxlabel(r"$\Delta\phi$", fontsize=24)
    fig.supylabel(r"$\Delta\Psi$", fontsize=24)
    
    plt.tight_layout()
    plt.show()

## 2x3 plot showing ROC, PRC, and scatterplot for a single feature

In [8]:
def plot_roc_prc_scatter(data, feature):
    """
    Creates a 2x3 plot showing ROC, PRC, and scatterplot for a single feature across both cell lines.
    
    Parameters:
    -----------
    data : DataFrame
        Input dataframe containing the data
    feature : str
        Feature name (e.g., 'RBFOX2_3_binding')
    """
    
    cell_lines = ['HepG2', 'K562']
    
    # Parse feature name (format: RBP_position_binding)
    parts = feature.split('_')
    rbp_name = parts[0]
    position = parts[1]
    parsed_title = f"{rbp_name} Position {position}"
    
    # Create 2x3 subplot figure
    fig, axes = plt.subplots(2, 3, figsize=(18, 12))
    
    for idx, cell_line in enumerate(cell_lines):
        
        # Subset for cell line and feature
        feature_subset = data[(data['cell_line'] == cell_line) & 
                             (data['Feature'] == feature)].copy()
        
        # Create Significance column based on dPSI and FDR values
        feature_subset['Significance'] = (
            (feature_subset['dPSI'].abs() > 0) & 
            (feature_subset['FDR'] <= 0.1)
        ).astype(int)
        
        # Calculate metrics
        total_events = len(feature_subset)
        num_significant = feature_subset['Significance'].sum()
        baseline = num_significant / total_events if total_events > 0 else 0
        
        # Check if we have both positive and negative examples
        if num_significant > 0 and num_significant < total_events:
            
            # Calculate AUROC and AUPRC
            auroc = roc_auc_score(feature_subset['Significance'], 
                                 feature_subset['CTRL - KD Local SHAP'].abs())
            auprc = average_precision_score(feature_subset['Significance'], 
                                           feature_subset['CTRL - KD Local SHAP'].abs())
            
            # Calculate curves
            fpr, tpr, _ = roc_curve(feature_subset['Significance'], 
                                   feature_subset['CTRL - KD Local SHAP'].abs())
            precision, recall, _ = precision_recall_curve(feature_subset['Significance'], 
                                                         feature_subset['CTRL - KD Local SHAP'].abs())
            
            # ROC curve (column 0)
            ax_roc = axes[idx, 0]
            ax_roc.plot(fpr, tpr, label=f'AUROC = {auroc:.3f}', linewidth=2)
            ax_roc.plot([0, 1], [0, 1], 'k--', label='Random', linewidth=2)
            ax_roc.plot([], [], ' ', label=f'Total Events: {total_events:,}')
            ax_roc.plot([], [], ' ', label=f'Significant Events: {num_significant:,}')
            ax_roc.set_xlabel('False Positive Rate', fontsize=14, labelpad=10)
            ax_roc.set_ylabel('True Positive Rate', fontsize=14, labelpad=10)
            ax_roc.set_title(f'ROC Curve', fontsize=16, pad=15)
            ax_roc.legend(fontsize=12)
            ax_roc.tick_params(axis='both', labelsize=12)
            
            # PRC curve (column 1)
            ax_prc = axes[idx, 1]
            ax_prc.plot(recall, precision, label=f'AUPRC = {auprc:.3f}', linewidth=2)
            ax_prc.axhline(y=baseline, color='k', linestyle='--', 
                          label=f'Baseline = {baseline:.3f}', linewidth=2)
            ax_prc.plot([], [], ' ', label=f'Total Events: {total_events:,}')
            ax_prc.plot([], [], ' ', label=f'Significant Events: {num_significant:,}')
            ax_prc.set_xlabel('Recall', fontsize=14, labelpad=10)
            ax_prc.set_ylabel('Precision', fontsize=14, labelpad=10)
            ax_prc.set_title(f'Precision-Recall Curve', fontsize=16, pad=15)
            ax_prc.legend(fontsize=12)
            ax_prc.tick_params(axis='both', labelsize=12)
            
        else:
            # Handle case with insufficient data for ROC and PRC
            for col in range(2):
                axes[idx, col].text(0.5, 0.5, 
                                   f'Insufficient data\n' + 
                                   f'Significant events: {num_significant}/{total_events}',
                                   ha='center', va='center', fontsize=12)
                if col == 0:
                    axes[idx, col].set_title(f'ROC Curve', fontsize=16, pad=15)
                else:
                    axes[idx, col].set_title(f'Precision-Recall Curve', fontsize=16, pad=15)
        
        # Scatterplot (column 2)
        plt.style.use("/project/PlatigLab/users/reece/ENCODE-RNA-Binding-Protein-Network-Modeling/analysis/07_reece_biological_validations/paper.mplstyle")
        ax_scatter = axes[idx, 2]
        
        if len(feature_subset) > 0:
            # Determine significance: blue if |dPSI| >= 0 AND FDR <= 0.1, else grey
            colors = ['tab:blue' if (abs(dpsi) > 0 and fdr <= 0.1) else 'lightgrey' 
                     for dpsi, fdr in zip(feature_subset["dPSI"], feature_subset["FDR"])]
            
            # Create the scatter plot with significance colors
            ax_scatter.scatter(feature_subset["CTRL - KD Local SHAP"], feature_subset["dPSI"], 
                      c=colors, alpha=0.5, s=30)
            
            # Calculate sign concordance for significant events only
            significant_subset = feature_subset[feature_subset['Significance'] == 1]
            if len(significant_subset) > 0:
                shap_values = significant_subset["CTRL - KD Local SHAP"]
                dpsi_values = significant_subset["dPSI"]
                same_sign = ((shap_values > 0) & (dpsi_values > 0)) | ((shap_values < 0) & (dpsi_values < 0))
                sign_concordance = (same_sign.sum() / len(significant_subset)) * 100
                
                # Add text box with sign concordance (bottom left, larger font)
                textstr = f'Sign concordance: {sign_concordance:.1f}%'
                props = dict(boxstyle='round', facecolor='white', edgecolor='black', alpha=0.8)
                ax_scatter.text(0.05, 0.05, textstr, transform=ax_scatter.transAxes, 
                              fontsize=14, verticalalignment='bottom', bbox=props)
        
        # Add quadrant lines at x=0 and y=0
        ax_scatter.axhline(y=0, color='black', linestyle='-', linewidth=1.5)
        ax_scatter.axvline(x=0, color='black', linestyle='-', linewidth=1.5)
        
        # Add grid
        ax_scatter.grid(True, alpha=0.3)
        
        # Labels and title
        ax_scatter.set_xlabel(r"$\Delta\varphi$", fontsize=14, labelpad=10) #(CTRL - KD Local SHAP)"
        ax_scatter.set_ylabel(r"$\Delta\Psi$", fontsize=14, labelpad=10)
        ax_scatter.set_title(r"$\Delta\varphi$ vs. $\Delta\Psi$" , fontsize=16, pad=15)
        ax_scatter.tick_params(axis='both', labelsize=12)
        
        # Add cell line label to the left of the first plot in each row
        axes[idx, 0].text(-0.25, 0.5, cell_line, 
                          transform=axes[idx, 0].transAxes,
                          fontsize=20, fontweight='bold',
                          rotation=90, va='center', ha='center')
    
    # Overall title with parsed feature name
    fig.suptitle(f'{parsed_title}', fontsize=18, y=0.995)
    plt.tight_layout()
    plt.show()

## 2x3 plot showing ROC, PRC, and scatterplot for all features combined

In [9]:
def plot_roc_prc_scatter_all_features(data):
    """
    Creates a 2x3 plot showing ROC, PRC, and scatterplot for ALL features
    combined across both cell lines. Requires >=10 significant events.
    """
    
    cell_lines = ['HepG2', 'K562']
    min_significant = 10

    fig, axes = plt.subplots(2, 3, figsize=(18, 12))
    
    for idx, cell_line in enumerate(cell_lines):
        
        # Subset for cell line (ALL features combined)
        feature_subset = data[data['cell_line'] == cell_line].copy()
        
        # Significance definition
        feature_subset['Significance'] = (
            (feature_subset['dPSI'].abs() > 0) & 
            (feature_subset['FDR'] <= 0.1)
        ).astype(int)
        
        total_events = len(feature_subset)
        num_significant = feature_subset['Significance'].sum()
        baseline = num_significant / total_events if total_events > 0 else 0
        
        # =======================
        # ROC + PRC
        # =======================
        if (
            num_significant >= min_significant and 
            num_significant < total_events
        ):
            
            scores = feature_subset['CTRL - KD Local SHAP'].abs()
            labels = feature_subset['Significance']
            
            auroc = roc_auc_score(labels, scores)
            auprc = average_precision_score(labels, scores)
            
            fpr, tpr, _ = roc_curve(labels, scores)
            precision, recall, _ = precision_recall_curve(labels, scores)
            
            # ROC
            ax_roc = axes[idx, 0]
            ax_roc.plot(fpr, tpr, label=f'AUROC = {auroc:.3f}', linewidth=2)
            ax_roc.plot([0, 1], [0, 1], 'k--', label='Random', linewidth=2)
            ax_roc.plot([], [], ' ', label=f'Total Events: {total_events:,}')
            ax_roc.plot([], [], ' ', label=f'Significant Events: {num_significant:,}')
            ax_roc.set_xlabel('False Positive Rate', fontsize=14, labelpad=10)
            ax_roc.set_ylabel('True Positive Rate', fontsize=14, labelpad=10)
            ax_roc.set_title('ROC Curve', fontsize=16, pad=15)
            ax_roc.legend(fontsize=12)
            ax_roc.tick_params(axis='both', labelsize=12)
            
            # PRC
            ax_prc = axes[idx, 1]
            ax_prc.plot(recall, precision, label=f'AUPRC = {auprc:.3f}', linewidth=2)
            ax_prc.axhline(
                y=baseline, 
                color='k', 
                linestyle='--',
                label=f'Baseline = {baseline:.3f}', 
                linewidth=2
            )
            ax_prc.plot([], [], ' ', label=f'Total Events: {total_events:,}')
            ax_prc.plot([], [], ' ', label=f'Significant Events: {num_significant:,}')
            ax_prc.set_xlabel('Recall', fontsize=14, labelpad=10)
            ax_prc.set_ylabel('Precision', fontsize=14, labelpad=10)
            ax_prc.set_title('Precision-Recall Curve', fontsize=16, pad=15)
            ax_prc.legend(fontsize=12)
            ax_prc.tick_params(axis='both', labelsize=12)
            
        else:
            for col in range(2):
                axes[idx, col].text(
                    0.5,
                    0.5,
                    f'Insufficient data\n'
                    f'Significant events: {num_significant}/{total_events}\n'
                    f'Min required: {min_significant}',
                    ha='center',
                    va='center',
                    fontsize=12
                )
                
                if col == 0:
                    axes[idx, col].set_title('ROC Curve', fontsize=16, pad=15)
                else:
                    axes[idx, col].set_title('Precision-Recall Curve', fontsize=16, pad=15)

        # =======================
        # Scatter
        # =======================
        plt.style.use("/project/PlatigLab/users/reece/ENCODE-RNA-Binding-Protein-Network-Modeling/analysis/07_reece_biological_validations/paper.mplstyle")
        ax_scatter = axes[idx, 2]
        
        if len(feature_subset) > 0:
            
            colors = [
                'tab:blue' if (abs(dpsi) > 0 and fdr <= 0.1)
                else 'lightgrey'
                for dpsi, fdr in zip(
                    feature_subset["dPSI"],
                    feature_subset["FDR"]
                )
            ]
            
            ax_scatter.scatter(
                feature_subset["CTRL - KD Local SHAP"],
                feature_subset["dPSI"],
                c=colors,
                alpha=0.5,
                s=30
            )
            
            # Sign concordance
            significant_subset = feature_subset[
                feature_subset['Significance'] == 1
            ]
            
            if len(significant_subset) > 0:
                
                shap_values = significant_subset["CTRL - KD Local SHAP"]
                dpsi_values = significant_subset["dPSI"]
                
                same_sign = (
                    ((shap_values > 0) & (dpsi_values > 0)) |
                    ((shap_values < 0) & (dpsi_values < 0))
                )
                
                sign_concordance = (
                    same_sign.sum() / len(significant_subset)
                ) * 100
                
                textstr = f'Sign concordance: {sign_concordance:.1f}%'
                props = dict(
                    boxstyle='round',
                    facecolor='white',
                    edgecolor='black',
                    alpha=0.8
                )
                
                ax_scatter.text(
                    0.05,
                    0.05,
                    textstr,
                    transform=ax_scatter.transAxes,
                    fontsize=14,
                    verticalalignment='bottom',
                    bbox=props
                )

        ax_scatter.axhline(y=0, color='black', linestyle='-', linewidth=1.5)
        ax_scatter.axvline(x=0, color='black', linestyle='-', linewidth=1.5)
        ax_scatter.grid(True, alpha=0.3)
        
        ax_scatter.set_xlabel(r"$\Delta\varphi$", fontsize=14, labelpad=10)
        ax_scatter.set_ylabel(r"$\Delta\Psi$", fontsize=14, labelpad=10)
        ax_scatter.set_title(
            r"$\Delta\varphi$ vs. $\Delta\Psi$",
            fontsize=16,
            pad=15
        )
        ax_scatter.tick_params(axis='both', labelsize=12)

        # Cell line label
        axes[idx, 0].text(
            -0.25,
            0.5,
            cell_line,
            transform=axes[idx, 0].transAxes,
            fontsize=20,
            fontweight='bold',
            rotation=90,
            va='center',
            ha='center'
        )

    fig.suptitle('All Features Combined', fontsize=18, y=0.995)
    plt.tight_layout()
    plt.show()

## 2x3 plot showing ROC, PRC, and scatterplot for all features in a given position

In [12]:
def plot_roc_prc_scatter_position(data, position):
    """
    Creates ROC, PRC, and scatter plots using ALL features at a given position.
    
    Example:
        plot_roc_prc_scatter_position(df, 6)
        -> uses all features like *_6_binding
    """
    
    cell_lines = ['HepG2', 'K562']
    min_significant = 10
    
    # match e.g. "_6_binding"
    position_pattern = f"_{position}_binding"
    
    fig, axes = plt.subplots(2, 3, figsize=(18, 12))
    
    for idx, cell_line in enumerate(cell_lines):
        
        # Filter cell line + position
        feature_subset = data[
            (data['cell_line'] == cell_line) &
            (data['Feature'].str.contains(position_pattern))
        ].copy()
        
        # Significance definition
        feature_subset['Significance'] = (
            (feature_subset['dPSI'].abs() > 0) &
            (feature_subset['FDR'] <= 0.1)
        ).astype(int)
        
        total_events = len(feature_subset)
        num_significant = feature_subset['Significance'].sum()
        baseline = num_significant / total_events if total_events > 0 else 0
        
        # ======================
        # ROC + PRC
        # ======================
        if num_significant >= min_significant and num_significant < total_events:
            
            scores = feature_subset['CTRL - KD Local SHAP'].abs()
            labels = feature_subset['Significance']
            
            auroc = roc_auc_score(labels, scores)
            auprc = average_precision_score(labels, scores)
            
            fpr, tpr, _ = roc_curve(labels, scores)
            precision, recall, _ = precision_recall_curve(labels, scores)
            
            # ROC
            ax_roc = axes[idx, 0]
            ax_roc.plot(fpr, tpr, label=f'AUROC = {auroc:.3f}', linewidth=2)
            ax_roc.plot([0, 1], [0, 1], 'k--', label='Random', linewidth=2)
            ax_roc.plot([], [], ' ', label=f'Total Events: {total_events:,}')
            ax_roc.plot([], [], ' ', label=f'Significant Events: {num_significant:,}')
            ax_roc.set_xlabel('False Positive Rate', fontsize=14, labelpad=10)
            ax_roc.set_ylabel('True Positive Rate', fontsize=14, labelpad=10)
            ax_roc.set_title('ROC Curve', fontsize=16, pad=15)
            ax_roc.legend(fontsize=12)
            
            # PRC
            ax_prc = axes[idx, 1]
            ax_prc.plot(recall, precision, label=f'AUPRC = {auprc:.3f}', linewidth=2)
            ax_prc.axhline(
                y=baseline,
                color='k',
                linestyle='--',
                label=f'Baseline = {baseline:.3f}',
                linewidth=2
            )
            ax_prc.plot([], [], ' ', label=f'Total Events: {total_events:,}')
            ax_prc.plot([], [], ' ', label=f'Significant Events: {num_significant:,}')
            ax_prc.set_xlabel('Recall', fontsize=14)
            ax_prc.set_ylabel('Precision', fontsize=14)
            ax_prc.set_title('Precision-Recall Curve', fontsize=16, pad=15)
            ax_prc.legend(fontsize=12)
            
        else:
            for col in range(2):
                axes[idx, col].text(
                    0.5,
                    0.5,
                    f'Insufficient data\n'
                    f'Significant events: {num_significant}/{total_events}\n'
                    f'Min required: {min_significant}',
                    ha='center',
                    va='center',
                    fontsize=12
                )

        # ======================
        # Scatter
        # ======================
        plt.style.use("/project/PlatigLab/users/reece/ENCODE-RNA-Binding-Protein-Network-Modeling/analysis/07_reece_biological_validations/paper.mplstyle")
        ax_scatter = axes[idx, 2]
        
        if len(feature_subset) > 0:
            
            colors = [
                'tab:blue' if (abs(dpsi) > 0 and fdr <= 0.1)
                else 'lightgrey'
                for dpsi, fdr in zip(
                    feature_subset["dPSI"],
                    feature_subset["FDR"]
                )
            ]
            
            ax_scatter.scatter(
                feature_subset["CTRL - KD Local SHAP"],
                feature_subset["dPSI"],
                c=colors,
                alpha=0.5,
                s=30
            )
            
            # sign concordance
            significant_subset = feature_subset[
                feature_subset['Significance'] == 1
            ]
            
            if len(significant_subset) > 0:
                shap_values = significant_subset["CTRL - KD Local SHAP"]
                dpsi_values = significant_subset["dPSI"]
                
                same_sign = (
                    ((shap_values > 0) & (dpsi_values > 0)) |
                    ((shap_values < 0) & (dpsi_values < 0))
                )
                
                sign_concordance = (
                    same_sign.sum() / len(significant_subset)
                ) * 100
                
                ax_scatter.text(
                    0.05,
                    0.05,
                    f'Sign concordance: {sign_concordance:.1f}%',
                    transform=ax_scatter.transAxes,
                    fontsize=14,
                    bbox=dict(
                        boxstyle='round',
                        facecolor='white',
                        edgecolor='black',
                        alpha=0.8
                    )
                )

        ax_scatter.axhline(0, color='black', linewidth=1.5)
        ax_scatter.axvline(0, color='black', linewidth=1.5)
        ax_scatter.grid(True, alpha=0.3)
        
        ax_scatter.set_xlabel(r"$\Delta\varphi$", fontsize=14)
        ax_scatter.set_ylabel(r"$\Delta\Psi$", fontsize=14)
        ax_scatter.set_title(
            r"$\Delta\varphi$ vs. $\Delta\Psi$",
            fontsize=16
        )

        # cell line label
        axes[idx, 0].text(
            -0.25,
            0.5,
            cell_line,
            transform=axes[idx, 0].transAxes,
            fontsize=20,
            fontweight='bold',
            rotation=90,
            va='center',
            ha='center'
        )

    fig.suptitle(f'All Features — Position {position}', fontsize=18, y=0.995)
    plt.tight_layout()
    plt.show()

## CSV that summarizes the ROC/PRC/Scatterplot info

In [8]:
# Makes a CSV that summarizes the ROC/PRC/Scatterplot info

def summarize_roc_prc_scatter(data):
    cell_lines = ['HepG2', 'K562']
    features = data['Feature'].unique()
    rows = []
    
    for feature in features:
        parts = feature.split('_')
        rbp_name = parts[0]
        position = parts[1]
        rbp_label = f"{rbp_name} Position {position}"
        
        for cell_line in cell_lines:
            
            feature_subset = data[(data['cell_line'] == cell_line) & 
                                  (data['Feature'] == feature)].copy()
            
            feature_subset['Significance'] = (
                (feature_subset['dPSI'].abs() > 0) & 
                (feature_subset['FDR'] <= 0.1)
            ).astype(int)
            
            total_events = len(feature_subset)
            num_significant = feature_subset['Significance'].sum()
            
            if num_significant > 0 and num_significant < total_events:
                auroc = roc_auc_score(feature_subset['Significance'],
                                      feature_subset['CTRL - KD Local SHAP'].abs())
                auprc = average_precision_score(feature_subset['Significance'],
                                                feature_subset['CTRL - KD Local SHAP'].abs())
            else:
                auroc = None
                auprc = None
            
            significant_subset = feature_subset[feature_subset['Significance'] == 1]
            if len(significant_subset) > 0:
                shap_values = significant_subset['CTRL - KD Local SHAP']
                dpsi_values = significant_subset['dPSI']
                same_sign = ((shap_values > 0) & (dpsi_values > 0)) | ((shap_values < 0) & (dpsi_values < 0))
                sign_concordance = (same_sign.sum() / len(significant_subset)) * 100
            else:
                sign_concordance = None
            
            rows.append({
                'RBP': rbp_label,
                'cell_line': cell_line,
                'Significant Events': num_significant,
                'AUROC': auroc,
                'AUPRC': auprc,
                'Sign Concordance': sign_concordance
            })
    
    return pd.DataFrame(rows).sort_values('Significant Events', ascending=False).reset_index(drop=True)

In [10]:
test2 = summarize_roc_prc_scatter(table)

In [11]:
test2.to_csv("updated_Crispr_candidate_info.csv")

In [18]:
column_mean = test2['AUROC'].mean()
column_median = test2['AUROC'].median()

# Print the results
print(f"The mean of ColumnA is: {column_mean}")
print(f"The median of ColumnA is: {column_median}")

The mean of ColumnA is: 0.5276317491124469
The median of ColumnA is: 0.5029104477611941
